# Intermediate 01 lab — Visual tokens to grounded generation

This notebook builds a deliberately small **local teaching VLM** for an industrial inspection scenario. It is not a foundation checkpoint and its scores are not claims about LLaVA, Qwen, Gemma, InternVL, or any public benchmark.

The lab keeps five evidence contracts separate:

1. answer quality by capability;
2. evidence-patch quality;
3. visual dependence under ablation and counterfactuals;
4. structured-output validity; and
5. systems cost under this notebook runtime.

All data are procedural, all default execution is offline, and optional foundation-model adapters are disabled unless you deliberately enable them.

## 1. Reproducibility and risk boundary

The seed and split policy are fixed before data generation. Factory A is training-only, Factory B is development-only, and Factory C is test-only. Development results may guide a decision; test results are reported once and never select prompts, thresholds, or model settings.

Generated language is advisory. A schema-valid answer, a high score, or a cited patch does not authorize a maintenance action.

In [ ]:
from __future__ import annotations

import json
import math
import os
import platform
import random
import re
import statistics
import sys
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import sklearn
import torch
from torch import nn
from torch.nn import functional as F

SEED = 17
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
DEVICE = torch.device("cpu")
print({"python": sys.version.split()[0], "torch": torch.__version__, "device": str(DEVICE), "seed": SEED})

## 2. Generate source-separated visual scenes

Each panel has one colored valve, a central pipe, two to four bolts, and an optional scratch. The scene graph is exact. Factory A is deliberately red-heavy, so a text-only baseline can exploit a language prior. Factories B and C change photometric conditions without changing the task ontology.

In [ ]:
IMAGE_SIZE = 64
PATCH_SIZE = 8
GRID = IMAGE_SIZE // PATCH_SIZE

def render_scene(scene, source):
    palettes = {
        "Factory A": ((36, 40, 46), (118, 126, 136)),
        "Factory B": ((40, 48, 54), (104, 125, 140)),
        "Factory C": ((50, 45, 43), (130, 116, 108)),
    }
    bg, metal = palettes[source]
    image = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), bg)
    draw = ImageDraw.Draw(image)
    draw.rectangle((29, 5, 35, 59), fill=metal)
    valve_x = 15 if scene["side"] == "left" else 49
    color = (220, 48, 55) if scene["color"] == "red" else (42, 105, 225)
    draw.ellipse((valve_x - 8, 22, valve_x + 8, 38), fill=color, outline=(235, 235, 235), width=1)
    draw.line((valve_x - 7, 30, valve_x + 7, 30), fill=(245, 245, 245), width=2)
    draw.line((valve_x, 23, valve_x, 37), fill=(245, 245, 245), width=2)
    bolt_slots = [(10, 10), (54, 10), (10, 54), (54, 54)]
    for x, y in bolt_slots[: scene["bolts"]]:
        draw.ellipse((x - 3, y - 3, x + 3, y + 3), fill=(238, 198, 44))
    if scene["scratched"]:
        draw.line((valve_x - 6, 25, valve_x + 6, 35), fill=(255, 255, 255), width=2)
    arr = np.asarray(image).astype(np.float32)
    rng = np.random.default_rng(scene["seed"])
    sigma = {"Factory A": 1.5, "Factory B": 4.0, "Factory C": 6.0}[source]
    arr = np.clip(arr + rng.normal(0, sigma, arr.shape), 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def build_split(source, count, red_probability, seed):
    rng = np.random.default_rng(seed)
    rows = []
    for index in range(count):
        scene = {
            "image_id": f"{source.lower().replace(' ', '-')}-{index:03d}",
            "source": source,
            "color": "red" if rng.random() < red_probability else "blue",
            "side": "left" if rng.random() < 0.5 else "right",
            "bolts": int(rng.integers(2, 5)),
            "scratched": bool(rng.random() < 0.45),
            "seed": int(seed * 1000 + index),
        }
        scene["image"] = render_scene(scene, source)
        rows.append(scene)
    return rows

splits = {
    "train": build_split("Factory A", 96, 0.82, 11),
    "development": build_split("Factory B", 36, 0.50, 23),
    "test": build_split("Factory C", 36, 0.50, 37),
}
split_summary = pd.DataFrame([
    {"split": name, "source": rows[0]["source"], "images": len(rows), "red_rate": np.mean([r["color"] == "red" for r in rows])}
    for name, rows in splits.items()
])
assert len({rows[0]["source"] for rows in splits.values()}) == 3
split_summary

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for axis, scene in zip(axes.flat, splits["train"][:4] + splits["test"][:4]):
    axis.imshow(scene["image"])
    axis.set_title(f'{scene["source"]}\n{scene["color"]}, {scene["bolts"]} bolts')
    axis.axis("off")
plt.tight_layout()

## 3. Patchify images and preserve spatial identity

The local visual encoder computes transparent patch statistics: mean RGB, grayscale variation, yellow/white fractions, and normalized coordinates. These are teaching features—not a pretrained vision encoder. A 2×2 spatial pool compresses 64 patch tokens to 16 tokens while keeping a coarse 4×4 grid.

In [ ]:
def patch_features(image, patch_size=PATCH_SIZE):
    arr = np.asarray(image).astype(np.float32) / 255.0
    tokens = []
    for gy, y in enumerate(range(0, IMAGE_SIZE, patch_size)):
        for gx, x in enumerate(range(0, IMAGE_SIZE, patch_size)):
            patch = arr[y:y+patch_size, x:x+patch_size]
            rgb = patch.mean(axis=(0, 1))
            gray_std = patch.mean(axis=2).std()
            yellow = ((patch[..., 0] > .65) & (patch[..., 1] > .55) & (patch[..., 2] < .45)).mean()
            white = (patch.mean(axis=2) > .86).mean()
            tokens.append([*rgb, gray_std, yellow, white, gx / (GRID - 1), gy / (GRID - 1)])
    return np.asarray(tokens, dtype=np.float32)

def spatial_pool(tokens):
    grid = tokens.reshape(GRID, GRID, -1)
    pooled = grid.reshape(4, 2, 4, 2, -1).mean(axis=(1, 3))
    return pooled.reshape(16, -1)

sample_full = patch_features(splits["train"][0]["image"])
sample_pooled = spatial_pool(sample_full)
print("full patch tokens:", sample_full.shape, "pooled visual tokens:", sample_pooled.shape)
assert sample_full.shape == (64, 8) and sample_pooled.shape == (16, 8)

## 4. Visual-token, attention, and KV-cache budgets

These calculations are explanatory components, **not total runtime memory**. Real processors may add thumbnails, tiles, special tokens, registers, padding, or token merging. Fused kernels can also avoid materializing naive attention intermediates even when the mathematical semantics are equivalent.

In [ ]:
def token_budget(image_size, patch_size=14, images=1, text_tokens=128):
    visual = math.ceil(image_size / patch_size) ** 2 * images
    total = visual + text_tokens
    return {
        "resolution": f"{image_size}×{image_size}", "images": images,
        "visual_tokens": visual, "total_context_tokens": total,
        "naive_attention_interactions": total ** 2,
    }

budget_table = pd.DataFrame([
    token_budget(224), token_budget(448), token_budget(768), token_budget(448, images=4)
])
budget_table

In [ ]:
def attention_and_kv_components(tokens, heads=16, layers=32, head_dim=128, bytes_per_value=2):
    attention_mib_one_layer = heads * tokens * tokens * bytes_per_value / 2**20
    kv_mib_all_layers = 2 * layers * tokens * heads * head_dim * bytes_per_value / 2**20
    return {"tokens": tokens, "attention_weights_MiB_one_layer": attention_mib_one_layer,
            "KV_cache_MiB_all_layers": kv_mib_all_layers}

memory_components = pd.DataFrame(attention_and_kv_components(n) for n in [197, 577, 2305, 4097])
memory_components

## 5. Executable connector patterns

Projection changes width; resampling changes length; cross-attention retains a separate visual memory. None of these operations alone guarantees semantic alignment.

In [ ]:
class LinearProjector(nn.Module):
    def __init__(self, d_vision=8, d_language=48):
        super().__init__(); self.net = nn.Linear(d_vision, d_language)
    def forward(self, visual): return self.net(visual)

class MLPProjector(nn.Module):
    def __init__(self, d_vision=8, d_language=48):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d_vision, 64), nn.GELU(), nn.Linear(64, d_language))
    def forward(self, visual): return self.net(visual)

class QueryResampler(nn.Module):
    def __init__(self, d_vision=8, d_model=48, queries=6, heads=4):
        super().__init__()
        self.kv = nn.Linear(d_vision, d_model)
        self.queries = nn.Parameter(torch.randn(1, queries, d_model) * .02)
        self.attn = nn.MultiheadAttention(d_model, heads, batch_first=True)
    def forward(self, visual):
        memory = self.kv(visual)
        q = self.queries.expand(visual.shape[0], -1, -1)
        return self.attn(q, memory, memory, need_weights=False)[0]

class CrossAttentionFusion(nn.Module):
    def __init__(self, d_vision=8, d_model=48, heads=4):
        super().__init__(); self.memory = nn.Linear(d_vision, d_model); self.attn = nn.MultiheadAttention(d_model, heads, batch_first=True)
    def forward(self, language, visual): return self.attn(language, self.memory(visual), self.memory(visual), need_weights=False)[0]

visual_batch = torch.tensor(np.stack([sample_pooled, sample_pooled]))
language_batch = torch.randn(2, 7, 48)
connector_shapes = {
    "linear": tuple(LinearProjector()(visual_batch).shape),
    "mlp": tuple(MLPProjector()(visual_batch).shape),
    "query_resampler": tuple(QueryResampler()(visual_batch).shape),
    "cross_attention_language": tuple(CrossAttentionFusion()(language_batch, visual_batch).shape),
}
connector_shapes

## 6. Build bounded questions and exact labels

The answer vocabulary is intentionally closed so exact normalization is sufficient. Open-ended language would require a separately calibrated semantic-review contract.

In [ ]:
QUESTION_TEMPLATES = {
    "recognition": "what color is the valve",
    "counting": "how many bolts are visible",
    "spatial": "where is the valve relative to the pipe",
    "defect": "is the valve scratched",
}
NUMBER_WORD = {2: "two", 3: "three", 4: "four"}

def answer_for(scene, capability):
    return {
        "recognition": scene["color"],
        "counting": NUMBER_WORD[scene["bolts"]],
        "spatial": scene["side"],
        "defect": "yes" if scene["scratched"] else "no",
    }[capability]

def object_patch(scene, capability):
    if capability == "counting":
        slots = [(10, 10), (54, 10), (10, 54), (54, 54)][:scene["bolts"]]
        return sorted({min(3, y // 16) * 4 + min(3, x // 16) for x, y in slots})
    x = 15 if scene["side"] == "left" else 49
    return [min(3, 30 // 16) * 4 + min(3, x // 16)]

def make_records(scenes, split):
    records = []
    for scene in scenes:
        visual = spatial_pool(patch_features(scene["image"]))
        for capability, question in QUESTION_TEMPLATES.items():
            records.append({"split": split, "scene": scene, "image_id": scene["image_id"],
                            "capability": capability, "question": question,
                            "answer": answer_for(scene, capability), "visual": visual,
                            "evidence_patches": object_patch(scene, capability)})
    return records

records = {name: make_records(rows, name) for name, rows in splits.items()}
pd.DataFrame([{k: r[k] for k in ["split", "image_id", "capability", "question", "answer"]}
              for r in records["train"]]).head(8)

In [ ]:
SPECIAL = ["<pad>", "<bos>", "<sep>", "<eos>"]
question_words = sorted(set(" ".join(QUESTION_TEMPLATES.values()).split()))
answer_words = ["red", "blue", "two", "three", "four", "left", "right", "yes", "no"]
VOCAB = SPECIAL + question_words + answer_words
stoi = {token: index for index, token in enumerate(VOCAB)}
itos = {index: token for token, index in stoi.items()}
PAD, BOS, SEP, EOS = [stoi[x] for x in SPECIAL]
MAX_Q = 10

def encode_question(text):
    ids = [BOS] + [stoi[word] for word in text.split()] + [SEP]
    sep_index = len(ids) - 1
    ids += [PAD] * (MAX_Q - len(ids))
    return ids[:MAX_Q], min(sep_index, MAX_Q - 1)

def tensorize(rows):
    questions, sep_positions = zip(*(encode_question(r["question"]) for r in rows))
    return (
        torch.tensor(np.stack([r["visual"] for r in rows]), dtype=torch.float32),
        torch.tensor(questions, dtype=torch.long),
        torch.tensor(sep_positions, dtype=torch.long),
        torch.tensor([stoi[r["answer"]] for r in rows], dtype=torch.long),
        torch.tensor([r["evidence_patches"][0] for r in rows], dtype=torch.long),
    )

## 7. Tiny causal multimodal generator

The model projects 16 visual tokens, prepends them to question tokens, and uses a causal transformer. The hidden state at `<sep>` predicts one bounded answer token; the next position predicts `<eos>`. A separate evidence head points to a coarse visual patch. Training loss is answer-token loss + EOS loss + evidence loss.

This is generative in the narrow autoregressive sense, but it is intentionally tiny and task-specific. It must not be described as a foundation model.

In [ ]:
class TinyGenerativeVLM(nn.Module):
    def __init__(self, vocab_size, d_model=48, layers=2, heads=4, visual_tokens=16):
        super().__init__()
        self.visual_tokens = visual_tokens
        self.connector = MLPProjector(8, d_model)
        self.text_embedding = nn.Embedding(vocab_size, d_model)
        self.position = nn.Parameter(torch.randn(1, visual_tokens + MAX_Q + 1, d_model) * .01)
        block = nn.TransformerEncoderLayer(d_model, heads, dim_feedforward=96, dropout=0.0, batch_first=True, norm_first=True)
        self.decoder = nn.TransformerEncoder(block, layers)
        self.lm_head = nn.Linear(d_model, vocab_size)
        self.evidence_key = nn.Linear(d_model, d_model)

    def forward(self, visual, questions, sep_positions, answer_tokens=None):
        v = self.connector(visual)
        text = self.text_embedding(questions)
        if answer_tokens is not None:
            text = torch.cat([text, self.text_embedding(answer_tokens[:, None])], dim=1)
        sequence = torch.cat([v, text], dim=1)
        sequence = sequence + self.position[:, :sequence.shape[1]]
        causal = torch.triu(torch.ones(sequence.shape[1], sequence.shape[1], device=sequence.device, dtype=torch.bool), diagonal=1)
        hidden = self.decoder(sequence, mask=causal)
        row = torch.arange(sequence.shape[0], device=sequence.device)
        sep_hidden = hidden[row, self.visual_tokens + sep_positions]
        answer_logits = self.lm_head(sep_hidden)
        eos_logits = None
        if answer_tokens is not None:
            eos_hidden = hidden[row, self.visual_tokens + MAX_Q]
            eos_logits = self.lm_head(eos_hidden)
        evidence_logits = torch.einsum("bd,bnd->bn", self.evidence_key(sep_hidden), v) / math.sqrt(v.shape[-1])
        return answer_logits, eos_logits, evidence_logits

    @torch.no_grad()
    def generate(self, visual, questions, sep_positions):
        answer_logits, _, evidence_logits = self(visual, questions, sep_positions)
        answer = answer_logits.argmax(dim=-1)
        _, eos_logits, _ = self(visual, questions, sep_positions, answer)
        return answer, eos_logits.argmax(dim=-1), evidence_logits.argmax(dim=-1), answer_logits.softmax(-1)

model = TinyGenerativeVLM(len(VOCAB)).to(DEVICE)
print("trainable parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
train_tensors = tuple(x.to(DEVICE) for x in tensorize(records["train"]))
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-3, weight_decay=1e-4)
history = []
for step in range(121):
    model.train(); optimizer.zero_grad()
    visual, questions, sep_positions, answers, evidence = train_tensors
    answer_logits, eos_logits, evidence_logits = model(visual, questions, sep_positions, answers)
    answer_loss = F.cross_entropy(answer_logits, answers)
    eos_loss = F.cross_entropy(eos_logits, torch.full_like(answers, EOS))
    evidence_loss = F.cross_entropy(evidence_logits, evidence)
    loss = answer_loss + 0.25 * eos_loss + 0.35 * evidence_loss
    loss.backward(); optimizer.step()
    if step % 20 == 0:
        history.append({"step": step, "loss": float(loss.detach()), "answer_loss": float(answer_loss.detach()), "evidence_loss": float(evidence_loss.detach())})
pd.DataFrame(history)

## 8. Capability-specific evaluation

Answers and evidence are scored separately. The evidence target may be set-valued (for example, any bolt patch is acceptable), even though training uses one deterministic representative patch.

In [ ]:
def evaluate(rows, label):
    model.eval()
    visual, questions, sep_positions, answers, _ = tensorize(rows)
    with torch.no_grad():
        predicted, eos, evidence, probabilities = model.generate(visual, questions, sep_positions)
    output = []
    for index, record in enumerate(rows):
        prediction = itos[int(predicted[index])]
        output.append({
            "split": label, "image_id": record["image_id"], "capability": record["capability"],
            "truth": record["answer"], "prediction": prediction,
            "answer_correct": prediction == record["answer"],
            "generated_eos": int(eos[index]) == EOS,
            "evidence_patch": int(evidence[index]),
            "evidence_hit": int(evidence[index]) in record["evidence_patches"],
            "confidence": float(probabilities[index, predicted[index]]),
        })
    frame = pd.DataFrame(output)
    summary = frame.groupby("capability").agg(answer_accuracy=("answer_correct", "mean"),
        evidence_hit_rate=("evidence_hit", "mean"), mean_confidence=("confidence", "mean")).reset_index()
    count_rows = frame[frame.capability == "counting"].copy()
    numeric = {"two": 2, "three": 3, "four": 4}
    count_rows["absolute_error"] = [abs(numeric.get(p, -9) - numeric[t]) for p, t in zip(count_rows.prediction, count_rows.truth)]
    summary["count_mae"] = np.nan
    summary.loc[summary.capability == "counting", "count_mae"] = count_rows.absolute_error.mean()
    return frame, summary

dev_predictions, dev_metrics = evaluate(records["development"], "development")
test_predictions, test_metrics = evaluate(records["test"], "test")
pd.concat([dev_metrics.assign(split="development"), test_metrics.assign(split="test")])

## 9. Language-prior baseline

This baseline never sees an image. It predicts the most common training answer for each capability. Strong accuracy on a skewed slice would not be visual intelligence; it would expose label priors.

In [ ]:
majority_answer = {}
for capability in QUESTION_TEMPLATES:
    majority_answer[capability] = Counter(r["answer"] for r in records["train"] if r["capability"] == capability).most_common(1)[0][0]

prior_rows = []
for record in records["test"]:
    prediction = majority_answer[record["capability"]]
    prior_rows.append({"capability": record["capability"], "correct": prediction == record["answer"]})
prior_metrics = pd.DataFrame(prior_rows).groupby("capability").correct.mean().rename("language_prior_accuracy").reset_index()
prior_metrics.merge(test_metrics, on="capability")

## 10. Visual Evidence Ablation and counterfactuals

The prompt and decoding policy stay fixed while the image changes. We record answer flips, answer probability, and evidence location. A correct answer under a blank image can be a lucky prior; a counterfactual that should change the answer is more diagnostic.

In [ ]:
def predict_one(image, question):
    visual = torch.tensor(spatial_pool(patch_features(image))[None], dtype=torch.float32)
    ids, sep = encode_question(question)
    with torch.no_grad():
        answer, eos, evidence, probs = model.generate(visual, torch.tensor([ids]), torch.tensor([sep]))
    token = itos[int(answer[0])]
    return token, float(probs[0, answer[0]]), int(evidence[0])

probe = splits["test"][0]
counterfactual = dict(probe)
counterfactual["color"] = "blue" if probe["color"] == "red" else "red"
counterfactual["seed"] += 100_000
wrong = next(scene for scene in splits["test"] if scene["color"] != probe["color"])
blank = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (42, 42, 42))
question = QUESTION_TEMPLATES["recognition"]

conditions = {
    "correct": probe["image"], "blank": blank, "wrong_image": wrong["image"],
    "counterfactual_color": render_scene(counterfactual, probe["source"]),
}
ablation_rows = []
base_answer = None
for condition, image in conditions.items():
    answer, probability, evidence_patch = predict_one(image, question)
    base_answer = answer if base_answer is None else base_answer
    expected = probe["color"] if condition in {"correct", "blank"} else (wrong["color"] if condition == "wrong_image" else counterfactual["color"])
    ablation_rows.append({"condition": condition, "expected_if_visible": expected, "answer": answer,
                          "answer_probability": probability, "evidence_patch": evidence_patch,
                          "answer_flip_vs_correct": answer != base_answer})
ablation_table = pd.DataFrame(ablation_rows)
ablation_table

In [ ]:
cf_row = ablation_table.query("condition == 'counterfactual_color'").iloc[0]
counterfactual_expected_flip = probe["color"] != counterfactual["color"]
counterfactual_observed_flip = bool(cf_row.answer_flip_vs_correct)
counterfactual_diagnostic = {
    "expected_answer_change": counterfactual_expected_flip,
    "observed_answer_change": counterfactual_observed_flip,
    "diagnosis": "visually responsive on this probe" if counterfactual_observed_flip else "failed visual counterfactual probe",
}
counterfactual_diagnostic

## 11. Evidence patches are evaluated—not treated as explanations

The predicted patch is an explicit auxiliary output. It can be correct while the answer is wrong, or wrong while the answer is correct. Neither the evidence head nor an attention map proves causal use of the pixels.

In [ ]:
joint_evidence = test_predictions.assign(
    outcome=np.select(
        [test_predictions.answer_correct & test_predictions.evidence_hit,
         test_predictions.answer_correct & ~test_predictions.evidence_hit,
         ~test_predictions.answer_correct & test_predictions.evidence_hit],
        ["answer correct + evidence correct", "answer correct + evidence wrong", "answer wrong + evidence correct"],
        default="answer wrong + evidence wrong"))
joint_evidence.groupby("outcome").size().rename("count").reset_index()

## 12. Structured-output validation and injected failures

Schema validation checks shape, field names, enums, and coordinate bounds. A second check compares the answer and patch against trusted labels. Keeping these layers separate prevents “valid JSON” from becoming shorthand for “true.”

In [ ]:
ABSTENTION_STATES = {"answerable", "insufficient_visual_evidence", "ambiguous_reference", "review_required"}

def patch_box(index):
    row, col = divmod(index, 4)
    return [col * 16, row * 16, (col + 1) * 16, (row + 1) * 16]

def validate_contract(payload, trusted_record=None):
    errors = []
    if set(payload) != {"answer", "capability", "abstention_state", "evidence"}:
        errors.append("top-level fields")
    if payload.get("capability") not in QUESTION_TEMPLATES: errors.append("capability enum")
    if payload.get("abstention_state") not in ABSTENTION_STATES: errors.append("abstention enum")
    evidence = payload.get("evidence", {})
    if set(evidence) != {"image_id", "patch_index", "box_xyxy"}: errors.append("evidence fields")
    box = evidence.get("box_xyxy", [])
    if len(box) != 4 or any(not isinstance(v, int) for v in box): errors.append("box shape/type")
    elif not (0 <= box[0] < box[2] <= IMAGE_SIZE and 0 <= box[1] < box[3] <= IMAGE_SIZE): errors.append("box bounds")
    if not isinstance(evidence.get("patch_index"), int) or not 0 <= evidence.get("patch_index", -1) < 16: errors.append("patch index")
    semantic = None
    if trusted_record is not None and not errors:
        semantic = {
            "answer_correct": payload["answer"] == trusted_record["answer"],
            "image_id_correct": evidence["image_id"] == trusted_record["image_id"],
            "evidence_hit": evidence["patch_index"] in trusted_record["evidence_patches"],
        }
    return {"schema_valid": not errors, "errors": errors, "semantic": semantic}

trusted = records["test"][0]
good = {"answer": trusted["answer"], "capability": trusted["capability"], "abstention_state": "answerable",
        "evidence": {"image_id": trusted["image_id"], "patch_index": trusted["evidence_patches"][0],
                     "box_xyxy": patch_box(trusted["evidence_patches"][0])}}
bad_extra = {**good, "authorized_action": "shutdown"}
bad_bounds = json.loads(json.dumps(good)); bad_bounds["evidence"]["box_xyxy"] = [-1, 0, 100, 100]
bad_evidence = json.loads(json.dumps(good)); bad_evidence["evidence"]["patch_index"] = (trusted["evidence_patches"][0] + 5) % 16
contract_tests = pd.DataFrame([
    {"case": "valid", **validate_contract(good, trusted)},
    {"case": "unknown action field", **validate_contract(bad_extra, trusted)},
    {"case": "out-of-bounds box", **validate_contract(bad_bounds, trusted)},
    {"case": "schema-valid wrong evidence", **validate_contract(bad_evidence, trusted)},
])
assert contract_tests.loc[contract_tests.case == "valid", "schema_valid"].item()
assert not contract_tests.loc[contract_tests.case == "unknown action field", "schema_valid"].item()
contract_tests

## 13. Deterministic before/after comparison proxy

This local proxy compares transparent patch statistics; it is **not the generative VLM and not a foundation model**. Its purpose is to establish a multi-image evaluation contract—change type, image attribution, and no-change behavior—before downloading a large checkpoint.

In [ ]:
def before_after_change_proxy(before, after):
    a, b = spatial_pool(patch_features(before)), spatial_pool(patch_features(after))
    red_blue_a, red_blue_b = (a[:, 0] - a[:, 2]).mean(), (b[:, 0] - b[:, 2]).mean()
    yellow_a, yellow_b = a[:, 4].sum(), b[:, 4].sum()
    white_a, white_b = a[:, 5].sum(), b[:, 5].sum()
    changes = {"color": abs(red_blue_b-red_blue_a), "bolt_count": abs(yellow_b-yellow_a), "scratch": abs(white_b-white_a)}
    change = max(changes, key=changes.get)
    if changes[change] < .015: change = "no_change"
    return {"change_type": change, "changed_image": "after" if change != "no_change" else "neither", "signals": changes}

pair_rows = []
for scene in splits["test"][:24]:
    modified = dict(scene); modified["seed"] += 200_000
    mode = ["color", "bolt_count", "scratch", "no_change"][len(pair_rows) % 4]
    if mode == "color": modified["color"] = "blue" if scene["color"] == "red" else "red"
    elif mode == "bolt_count": modified["bolts"] = 2 if scene["bolts"] != 2 else 4
    elif mode == "scratch": modified["scratched"] = not scene["scratched"]
    result = before_after_change_proxy(scene["image"], render_scene(modified, scene["source"]))
    pair_rows.append({"truth": mode, "prediction": result["change_type"], "image_attribution": result["changed_image"]})
comparison_results = pd.DataFrame(pair_rows)
comparison_summary = {"change_type_accuracy": float((comparison_results.truth == comparison_results.prediction).mean()),
                      "image_attribution_accuracy": float(((comparison_results.truth == "no_change") == (comparison_results.image_attribution == "neither")).mean())}
comparison_summary

## 14. Connector systems microbenchmark

Timing individual calls gives a distribution rather than a p95 of batch averages. These are demonstration measurements for this CPU runtime only; they are not service-level targets and do not include image decoding, a vision backbone, language prefill, or token decoding.

In [ ]:
def profile_module(name, module, visual, repeats=31):
    module.eval()
    with torch.no_grad():
        for _ in range(5): module(visual)
        samples = []
        for _ in range(repeats):
            start = time.perf_counter(); output = module(visual); samples.append((time.perf_counter() - start) * 1000)
    return {"connector": name, "output_tokens": output.shape[1],
            "parameters": sum(p.numel() for p in module.parameters()),
            "median_ms": statistics.median(samples), "p95_ms": float(np.percentile(samples, 95)),
            "iqr_ms": float(np.percentile(samples, 75) - np.percentile(samples, 25))}

profile_input = visual_batch[:1]
connector_profile = pd.DataFrame([
    profile_module("linear projection", LinearProjector(), profile_input),
    profile_module("MLP projection", MLPProjector(), profile_input),
    profile_module("query resampler (6 tokens)", QueryResampler(), profile_input),
])
connector_profile

## 15. Optional pinned foundation-model adapters

The disabled cell below demonstrates the common Transformers SDK path: `AutoProcessor`, a multimodal chat template, `AutoModelForImageTextToText`, and `generate`. It requires an explicit opt-in because checkpoints are large and runtime support changes. `trust_remote_code=False` remains the default.

Pinned revisions make a smoke test more reproducible, but a Git/model revision alone is not complete production provenance. Record resolved file hashes, license acceptance, processor files, quantization, runtime/container, and approvals before comparison or deployment.

In [ ]:
ENABLE_OPTIONAL_FOUNDATION_MODEL = os.getenv("CV_ENABLE_VLM", "0") == "1"
OPTIONAL_MODEL_MANIFEST = {
    "default_smoke_test": {"model_id": "HuggingFaceTB/SmolVLM-256M-Instruct", "revision": "7e3e67edbbed1bf9888184d9df282b700a323964"},
    "qwen3_vl_case": {"model_id": "Qwen/Qwen3-VL-2B-Instruct", "revision": "89644892e4d85e24eaac8bacfd4f463576704203"},
    "source_revisions_reviewed": {
        "QwenLM/Qwen3-VL": "96588727e44c78b25ba03ea03b8e12f7e64fd0da",
        "haotian-liu/LLaVA": "c121f0432da27facab705978f83c4ada465e46fd",
        "OpenGVLab/InternVL": "2410d1dbf208f0e799459aff9376e5747dbf41a2",
        "huggingface/transformers": "c93057d4835cd31752bb56f59989dd27696eb45b",
    },
    "artifact_hash_status": "not_resolved_by_default_notebook",
    "production_provenance_complete": False,
    "comparison_eligible": False,
}

optional_observation = {"enabled": False, "status": "not run; set CV_ENABLE_VLM=1 after reviewing license, storage, and runtime"}
if ENABLE_OPTIONAL_FOUNDATION_MODEL:
    from transformers import AutoModelForImageTextToText, AutoProcessor
    selected = OPTIONAL_MODEL_MANIFEST["default_smoke_test"]
    processor = AutoProcessor.from_pretrained(selected["model_id"], revision=selected["revision"], trust_remote_code=False)
    foundation_model = AutoModelForImageTextToText.from_pretrained(selected["model_id"], revision=selected["revision"], trust_remote_code=False)
    messages = [{"role": "user", "content": [{"type": "image", "image": splits["test"][0]["image"]},
                                                 {"type": "text", "text": "What color is the valve? Answer with one word."}]}]
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
    generated = foundation_model.generate(**inputs, max_new_tokens=8, do_sample=False)
    optional_observation = {"enabled": True, "decoded": processor.batch_decode(generated, skip_special_tokens=True),
                            "production_provenance_complete": False, "comparison_eligible": False}
optional_observation

## 16. Enterprise decision artifact

The artifact keeps local measurements, optional observations, and unresolved production assumptions in different fields. Demonstration thresholds are explicitly labeled as notebook-runtime teaching defaults.

In [ ]:
runtime_manifest = {
    "python": sys.version.split()[0], "platform": platform.platform(), "torch": torch.__version__,
    "numpy": np.__version__, "pandas": pd.__version__, "pillow": Image.__version__, "scikit_learn": sklearn.__version__,
    "device": str(DEVICE), "seed": SEED,
}
decision = {
    "course": "Intermediate 01 — Vision-Language Models",
    "evidence_scope": "local teaching model; no public benchmark or foundation-checkpoint claim",
    "split_contract": {"train": "Factory A", "development": "Factory B", "test": "Factory C"},
    "locally_measured_evidence": {
        "development_by_capability": dev_metrics.to_dict(orient="records"),
        "test_by_capability": test_metrics.to_dict(orient="records"),
        "language_prior_test": prior_metrics.to_dict(orient="records"),
        "visual_evidence_ablation": ablation_table.to_dict(orient="records"),
        "counterfactual_diagnostic": counterfactual_diagnostic,
        "comparison_proxy": comparison_summary,
        "connector_profile": connector_profile.to_dict(orient="records"),
    },
    "optional_downloaded_model_observations": optional_observation,
    "optional_model_manifest": OPTIONAL_MODEL_MANIFEST,
    "demonstration_thresholds_for_this_notebook_runtime_only": {
        "minimum_test_answer_accuracy_by_capability": 0.60,
        "maximum_connector_p95_ms": 50.0,
    },
    "unresolved_production_assumptions": [
        "representative governed domain data and rare/failure slices",
        "approved model and dataset licenses with resolved artifact hashes",
        "processor parity, target-hardware latency, memory, concurrency, and cost",
        "calibrated abstention and human-review capacity",
        "privacy, retention, tenant isolation, authorization, monitoring, and rollback",
    ],
    "runtime": runtime_manifest,
}
artifact_dir = Path("artifacts"); artifact_dir.mkdir(exist_ok=True)
artifact_path = artifact_dir / "intermediate-01-vlm-evidence.json"
artifact_path.write_text(json.dumps(decision, indent=2), encoding="utf-8")
print(artifact_path, "written; production readiness: not established")

## 17. Review prompts

- Which test questions were answerable from language priors alone?
- Did the model's counterfactual answer change when the relevant visual fact changed?
- How often was the answer correct while the cited evidence was wrong?
- What information did 4×4 pooling discard compared with the 8×8 patch grid?
- Which connector reduced token count, and what capability evidence would justify that compression?
- Which fields in the final artifact are measured, optional, and still unresolved?

Do not “fix” a failed diagnostic by hiding it. A failed ablation is evidence that the current design should not be trusted for that capability.